# LeetCode 804 — Unique Morse Code Words bằng Python

**Link đề:** https://leetcode.com/problems/unique-morse-code-words/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, String

```python
class Solution:
    def uniqueMorseRepresentations(self, words: list[str]) -> int:
        ...
```

## Mục tiêu học
1. Dùng **bảng tra** (lookup table) là một list 26 phần tử, truy cập bằng `ord(ch) - ord('a')`.
2. Ghép chuỗi hiệu quả bằng `"".join(...)` thay vì `+=` trong vòng lặp.
3. Đếm số giá trị **khác nhau** bằng `set` (thay vì list + kiểm tra `in` chậm).
4. Biết thêm các cách: `dict` + `zip`, `str.translate` — và so sánh ưu nhược.
5. Trace, chứng minh, phân tích Time/Space, luyện các biến thể (nhóm từ theo mã, giải mã).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Mã Morse quốc tế ánh xạ mỗi chữ cái thành một dãy chấm `.` và gạch `-`: `'a' → ".-"`, `'b' → "-..."`, `'c' → "-.-."`, ... Bảng đầy đủ cho 26 chữ:

```text
[".-","-...","-.-.","-..",".","..-.","--.","....","..",".---","-.-",".-..","--",
 "-.","---",".--.","--.-",".-.","...","-","..-","...-",".--","-..-","-.--","--.."]
```

Mỗi từ được viết bằng cách **nối** mã của từng chữ. Ví dụ `"cab"` → `"-.-." + ".-" + "-..."` = `"-.-..--..."`. Ta gọi đó là **phép biến đổi** (transformation) của từ.

Cho mảng `words`, trả về **số phép biến đổi khác nhau**.

**Ví dụ 1:** `["gin","zen","gig","msg"]` → `2`
```text
"gin" -> "--...-."
"zen" -> "--...-."
"gig" -> "--...--."
"msg" -> "--...--."
```
**Ví dụ 2:** `["a"]` → `1`

**Constraints:** `1 <= words.length <= 100`; `1 <= words[i].length <= 12`; chỉ chữ thường.

Điểm thú vị: **hai từ khác nhau có thể cho cùng mã** vì Morse ở đây **không có dấu cách** giữa các chữ.

In [1]:
MORSE = [".-","-...","-.-.","-..",".","..-.","--.","....","..",".---","-.-",".-..","--",
         "-.","---",".--.","--.-",".-.","...","-","..-","...-",".--","-..-","-.--","--.."]
print(len(MORSE), "mã")
print("a ->", MORSE[0], "| b ->", MORSE[1], "| z ->", MORSE[25])

26 mã
a -> .- | b -> -... | z -> --..


## 2. Nền tảng Python: từ chữ cái sang chỉ số bằng `ord`

`ord(ch)` trả về mã Unicode của ký tự. Các chữ thường liên tiếp nhau: `ord('a') = 97`, `ord('b') = 98`, ..., `ord('z') = 122`.

→ Chỉ số trong bảng: `ord(ch) - ord('a')` cho `a → 0`, `b → 1`, ..., `z → 25`. Ngược lại `chr(97 + i)` cho chữ thứ `i`.

In [2]:
for ch in "abgz":
    idx = ord(ch) - ord("a")
    print(ch, "ord =", ord(ch), "-> chỉ số", idx, "-> mã", MORSE[idx])
print("chr(97 + 6) =", chr(97 + 6))

a ord = 97 -> chỉ số 0 -> mã .-
b ord = 98 -> chỉ số 1 -> mã -...
g ord = 103 -> chỉ số 6 -> mã --.
z ord = 122 -> chỉ số 25 -> mã --..
chr(97 + 6) = g


### 2.1. Ghép chuỗi: `+=` và `"".join`

Hai cách tạo `"-.-..--..."` từ `"cab"`:
- Cộng dần `code += MORSE[...]`: dễ hiểu; mỗi lần có thể tạo chuỗi mới (chuỗi bất biến).
- `"".join(list_các_mảnh)`: tạo chuỗi kết quả **một lần**, là cách Pythonic.

In [3]:
word = "cab"
code = ""
for ch in word:
    piece = MORSE[ord(ch) - ord("a")]
    code += piece
    print(f"thêm {ch!r} -> {piece!r:8} => code = {code!r}")
pieces = [MORSE[ord(ch) - ord("a")] for ch in word]
print(pieces, "->", "".join(pieces))

thêm 'c' -> '-.-.'   => code = '-.-.'
thêm 'a' -> '.-'     => code = '-.-..-'
thêm 'b' -> '-...'   => code = '-.-..--...'
['-.-.', '.-', '-...'] -> -.-..--...


## 3. Hàm biến đổi một từ

Gói việc trên thành hàm `to_morse(word)` — khối xây dựng của mọi lời giải bên dưới.

In [4]:
def to_morse(word: str) -> str:
    return "".join(MORSE[ord(ch) - ord("a")] for ch in word)

for w in ["gin", "zen", "gig", "msg", "a", "cab"]:
    print(f"{w:>4} -> {to_morse(w)}")

 gin -> --...-.
 zen -> --...-.
 gig -> --...--.
 msg -> --...--.
   a -> .-
 cab -> -.-..--...


### 3.1. Vì sao hai từ khác nhau lại trùng mã? — mã **không phải** prefix-free

Một bộ mã gọi là **prefix-free** (không tiền tố) nếu không mã nào là **tiền tố** (phần đầu) của mã khác; khi đó chuỗi nối liền luôn tách ngược được duy nhất. Morse **không** prefix-free: `e = "."` là tiền tố của `a = ".-"`, `i = ".."`, ... Vì vậy bỏ dấu cách đi thì `"et"` và `"a"` cùng thành `".-"`. Đây chính là lý do bài toán có nghĩa.

`s.startswith(t)` trả `True` nếu chuỗi `s` bắt đầu bằng `t`.

In [5]:
print(".-".startswith("."), "--.".startswith(".-"))
prefix_pairs = []
for i, a in enumerate(MORSE):
    for j, c in enumerate(MORSE):
        if i != j and c.startswith(a):
            prefix_pairs.append((chr(97 + i), chr(97 + j)))
print("số cặp (x, y) mà mã x là tiền tố của mã y:", len(prefix_pairs))
print("vài ví dụ:", prefix_pairs[:8])

True False
số cặp (x, y) mà mã x là tiền tố của mã y: 56
vài ví dụ: [('a', 'j'), ('a', 'l'), ('a', 'p'), ('a', 'r'), ('a', 'w'), ('d', 'b'), ('d', 'x'), ('e', 'a')]


## 4. Cách 1 — Trực giác: list các mã đã thấy

Duyệt từng từ, biến đổi; nếu mã **chưa có** trong list `seen` thì thêm vào. Cuối cùng trả `len(seen)`.

Vấn đề: `code not in seen` với **list** phải duyệt cả list → `O(số mã đã thấy)` mỗi lần.

In [6]:
def unique_morse_list(words: list[str]) -> int:
    seen = []
    for w in words:
        code = to_morse(w)
        if code not in seen:
            seen.append(code)
    return len(seen)

print(unique_morse_list(["gin", "zen", "gig", "msg"]), unique_morse_list(["a"]))

2 1


**Độ phức tạp cách 1:** gọi `W` = số từ, `L` = độ dài tối đa của từ (mỗi mã dài ≤ `4L`). Kiểm tra `in` list: `O(W)` lần so sánh, mỗi lần so chuỗi `O(L)` → tổng `O(W² · L)`.

## 5. Nút thắt và insight — "đếm số giá trị khác nhau" = `len(set(...))`

`set` tự **loại trùng**: thêm một phần tử đã có thì không thay đổi. Tra/thêm trung bình `O(1)` (cộng chi phí băm chuỗi `O(L)`).

→ Chỉ cần đưa mọi mã vào set rồi lấy `len`.

In [7]:
s = set()
for code in ["--...-.", "--...-.", "--...--.", "--...--."]:
    s.add(code)
    print("add", code, "->", s)
print("số mã khác nhau:", len(s))
print(len(set([1, 1, 2, 3, 3, 3])))

add --...-. -> {'--...-.'}
add --...-. -> {'--...-.'}
add --...--. -> {'--...-.', '--...--.'}
add --...--. -> {'--...-.', '--...--.'}
số mã khác nhau: 2
3


## 6. Lời giải chính — `set` các phép biến đổi

```python
codes = set()                     # (1) tập mã khác nhau
for w in words:                   # (2) mỗi từ
    codes.add(to_morse(w))        # (3) biến đổi và thêm (trùng thì bỏ qua)
return len(codes)                 # (4) số phần tử = số mã khác nhau
```

In [8]:
def unique_morse(words: list[str]) -> int:
    codes = set()
    for w in words:
        codes.add(to_morse(w))
    return len(codes)

print(unique_morse(["gin", "zen", "gig", "msg"]), unique_morse(["a"]))

2 1


## 7. TRACE từng bước

| bước | từ | mã | đã có trong set? | set sau | len |
|---|---|---|:---:|---|---|
| 1 | gin | `--...-.` | ✘ | {`--...-.`} | 1 |
| 2 | zen | `--...-.` | ✔ | không đổi | 1 |
| 3 | gig | `--...--.` | ✘ | {`--...-.`, `--...--.`} | 2 |
| 4 | msg | `--...--.` | ✔ | không đổi | 2 |

Trace in thêm từng mảnh của mỗi chữ để thấy vì sao `gin` và `zen` trùng: `g i n = --. + .. + -.` và `z e n = --.. + . + -.` — ranh giới chữ khác nhau nhưng chuỗi nối giống hệt.

In [9]:
def trace_unique_morse(words):
    codes = set()
    for step, w in enumerate(words, start=1):
        pieces = [MORSE[ord(ch) - ord("a")] for ch in w]
        code = "".join(pieces)
        existed = code in codes
        codes.add(code)
        print(f"bước {step}: {w!r:6} = {' + '.join(pieces):<18} = {code:<10} "
              f"{'(đã có)' if existed else '(MỚI) '} | len(set) = {len(codes)}")
    print("Kết quả:", len(codes))
    return len(codes)

_ = trace_unique_morse(["gin", "zen", "gig", "msg"])

bước 1: 'gin'  = --. + .. + -.      = --...-.    (MỚI)  | len(set) = 1
bước 2: 'zen'  = --.. + . + -.      = --...-.    (đã có) | len(set) = 1
bước 3: 'gig'  = --. + .. + --.     = --...--.   (MỚI)  | len(set) = 2
bước 4: 'msg'  = -- + ... + --.     = --...--.   (đã có) | len(set) = 2
Kết quả: 2


`enumerate(words, start=1)` đếm bước từ 1 thay vì 0 — tiện cho việc in.

## 8. Code nộp LeetCode — lời giải chính

In [10]:
class Solution:
    def uniqueMorseRepresentations(self, words: list[str]) -> int:
        morse = [".-","-...","-.-.","-..",".","..-.","--.","....","..",".---","-.-",".-..","--",
                 "-.","---",".--.","--.-",".-.","...","-","..-","...-",".--","-..-","-.--","--.."]
        return len({"".join(morse[ord(c) - ord("a")] for c in w) for w in words})

sol = Solution()
assert sol.uniqueMorseRepresentations(["gin", "zen", "gig", "msg"]) == 2
assert sol.uniqueMorseRepresentations(["a"]) == 1
print("Solution OK")

Solution OK


Dòng `return` dùng **set comprehension** `{biểu_thức for w in words}` — tạo set trực tiếp, gọn như list comprehension nhưng dùng ngoặc nhọn.

## 9. Cách 3 — `dict` ánh xạ chữ → mã bằng `zip`

Thay vì tính chỉ số bằng `ord`, tạo dict `{'a': '.-', 'b': '-...', ...}` bằng `zip(string.ascii_lowercase, MORSE)` rồi `dict(...)`.

In [11]:
import string
print(string.ascii_lowercase)
pairs = list(zip("abc", MORSE))
print(pairs)
CODE = dict(zip(string.ascii_lowercase, MORSE))
print(CODE["g"], CODE["z"])

def unique_morse_dict(words):
    return len({"".join(CODE[c] for c in w) for w in words})

print(unique_morse_dict(["gin", "zen", "gig", "msg"]))

abcdefghijklmnopqrstuvwxyz
[('a', '.-'), ('b', '-...'), ('c', '-.-.')]
--. --..
2


### 9.1. Duyệt dict bằng `.items()` và **dict comprehension**

- `d.items()` cho các cặp `(key, value)`.
- `{v: k for k, v in d.items()}` là **dict comprehension** — ở đây dùng để **đảo** dict (mã → chữ), sẽ cần khi muốn **giải mã** ngược lại (bài tập 5). Đảo được an toàn vì 26 mã Morse đôi một khác nhau.

In [12]:
for k, v in list(CODE.items())[:3]:
    print(k, "->", v)
REV_DEMO = {v: k for k, v in CODE.items()}
print(len(REV_DEMO), "mã khác nhau ->", REV_DEMO[".-"], REV_DEMO["--.."])

a -> .-
b -> -...
c -> -.-.
26 mã khác nhau -> a z


## 10. Cách 4 — `str.translate`: biến đổi cả từ trong một lệnh

`str.maketrans(d)` nhận dict **ký tự → chuỗi** và tạo bảng dịch; `w.translate(table)` thay mỗi ký tự theo bảng — chạy trong C nên rất nhanh. Đây là lời giải "sáng tạo" ít người nghĩ tới.

In [13]:
TABLE = str.maketrans(CODE)
print("cab".translate(TABLE))
print("hello".translate(str.maketrans({"l": "L", "o": "0"})))

def unique_morse_translate(words):
    return len({w.translate(TABLE) for w in words})

print(unique_morse_translate(["gin", "zen", "gig", "msg"]))

-.-..--...
heLL0
2


In [14]:
class Solution:
    def uniqueMorseRepresentations(self, words: list[str]) -> int:
        morse = [".-","-...","-.-.","-..",".","..-.","--.","....","..",".---","-.-",".-..","--",
                 "-.","---",".--.","--.-",".-.","...","-","..-","...-",".--","-..-","-.--","--.."]
        table = str.maketrans(dict(zip("abcdefghijklmnopqrstuvwxyz", morse)))
        return len({w.translate(table) for w in words})

print(Solution().uniqueMorseRepresentations(["gin", "zen", "gig", "msg"]))

2


## 11. Vì sao đúng?

- `to_morse` cài đúng định nghĩa: nối mã từng chữ theo thứ tự.
- **Invariant:** sau khi xử lý `k` từ đầu, `codes` = tập **đúng** các mã khác nhau của `k` từ đó (set chứa mỗi giá trị tối đa một lần, và mọi mã đã gặp đều được `add`).
- Hết vòng ⇒ `len(codes)` = số phép biến đổi khác nhau.

Lưu ý: ta đếm **mã khác nhau**, không phải **từ khác nhau** — hai từ khác nhau (`gin`, `zen`) cùng mã chỉ tính 1; hai từ giống nhau đương nhiên cùng mã.

## 12. Độ phức tạp và so sánh

`W` = số từ (≤ 100), `L` = độ dài từ (≤ 12); mỗi mã dài ≤ `4L`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. List + `not in` | `O(W² · L)` | `O(W · L)` | Chậm khi nhiều từ |
| 2. Set (chính) | `O(W · L)` trung bình | `O(W · L)` | **Khuyến nghị** |
| 3. Dict + `zip` | `O(W · L)` | `O(W · L)` | Đọc dễ, không cần `ord` |
| 4. `str.translate` | `O(W · L)` | `O(W · L)` | Nhanh nhất thực tế (C) |

In [15]:
import random, time
random.seed(804)
big = ["".join(random.choice("abcde") for _ in range(random.randint(1, 12))) for _ in range(20000)]
for f in [unique_morse_list, unique_morse, unique_morse_dict, unique_morse_translate]:
    t = time.perf_counter()
    r = f(big)
    print(f"{f.__name__:<24} -> {r}  ({(time.perf_counter() - t) * 1000:.1f} ms)")

unique_morse_list        -> 12858  (595.7 ms)
unique_morse             -> 12858  (16.8 ms)
unique_morse_dict        -> 12858  (12.9 ms)
unique_morse_translate   -> 12858  (8.1 ms)


## 13. Bộ test

1. Hai ví dụ đề.
2. Một từ.
3. Các từ giống hệt nhau → 1.
4. Từ khác nhau nhưng cùng mã (`"gin"`/`"zen"`; `"et"` = `"." + "-"` và `"a"` = `".-"`).
5. Tất cả mã khác nhau (26 chữ cái đơn).
6. Đối chiếu 4 cách trên dữ liệu ngẫu nhiên.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

check("đề 1", unique_morse(["gin", "zen", "gig", "msg"]), 2)
check("đề 2", unique_morse(["a"]), 1)
check("trùng từ", unique_morse(["abc", "abc", "abc"]), 1)
check("'et' và 'a' cùng mã", unique_morse(["et", "a"]), 1)
check("26 chữ đơn", unique_morse(list(string.ascii_lowercase)), 26)
print("to_morse('et') =", to_morse("et"), "| to_morse('a') =", to_morse("a"))

PASS đề 1: 2
PASS đề 2: 1
PASS trùng từ: 1
PASS 'et' và 'a' cùng mã: 1
PASS 26 chữ đơn: 26
to_morse('et') = .- | to_morse('a') = .-


In [17]:
for _ in range(300):
    ws = ["".join(random.choice("aeitmn") for _ in range(random.randint(1, 4))) for _ in range(random.randint(1, 30))]
    expected = unique_morse_list(ws)
    assert unique_morse(ws) == expected
    assert unique_morse_dict(ws) == expected
    assert unique_morse_translate(ws) == expected
    assert Solution().uniqueMorseRepresentations(ws) == expected
print("300 bộ ngẫu nhiên (chữ a,e,i,t,m,n có mã ngắn -> nhiều va chạm): 5 cách khớp nhau")

300 bộ ngẫu nhiên (chữ a,e,i,t,m,n có mã ngắn -> nhiều va chạm): 5 cách khớp nhau


## 14. Lỗi người mới hay gặp

### Lỗi 1: Đếm số **từ** khác nhau
```python
return len(set(words))   # SAI: "gin" và "zen" là 2 từ nhưng 1 mã
```

### Lỗi 2: Chèn dấu cách giữa các mã
```python
" ".join(MORSE[...] for c in w)   # SAI: đề nối liền, có dấu cách thì 'gin' và 'zen' khác nhau
```

### Lỗi 3: Lệch chỉ số
```python
MORSE[ord(c) - ord("a") + 1]   # SAI: 'a' lấy mã của 'b'; 'z' -> IndexError
```

### Lỗi 4: `set(code)` thay vì `{code}` / `add`
`set("--.")` tách chuỗi thành tập **ký tự** `{'-', '.'}`, không phải tập chứa một chuỗi.

In [18]:
words1 = ["gin", "zen", "gig", "msg"]
print("len(set(words)) =", len(set(words1)), "| đúng =", unique_morse(words1))
print("có dấu cách     =", len({" ".join(MORSE[ord(c) - 97] for c in w) for w in words1}))
print("set('--.')      =", set("--."), "| {'--.'} =", {"--."})
try:
    MORSE[ord("z") - ord("a") + 1]
except IndexError as e:
    print("lệch chỉ số với 'z':", type(e).__name__, e)

len(set(words)) = 4 | đúng = 2
có dấu cách     = 4
set('--.')      = {'-', '.'} | {'--.'} = {'--.'}
lệch chỉ số với 'z': IndexError list index out of range


## 15. Mẹo nhận dạng pattern

- "Có bao nhiêu **giá trị khác nhau**" ⇒ `len(set(...))`.
- "Mỗi chữ cái ánh xạ sang ..." với bảng cho sẵn 26 phần tử ⇒ **lookup table** `table[ord(c) - ord('a')]` hoặc `dict(zip(...))`.
- "Biến đổi rồi so sánh / gom nhóm" ⇒ tính **khóa chuẩn hóa** (canonical key) cho mỗi phần tử rồi dùng set/dict. (Cùng kỹ thuật với Group Anagrams: khóa = chuỗi đã sắp xếp.)

### Biến thể

1. **Nhóm** các từ theo mã (dict: mã → list từ).
2. Tìm mã **phổ biến nhất**.
3. Morse **có dấu cách** giữa chữ và **giải mã** ngược lại thành từ.
4. Đếm số **cặp từ** có cùng mã.
5. LeetCode 49 Group Anagrams, 1078... — cùng tư duy "khóa chuẩn hóa".

In [19]:
# Biến thể 1: nhóm từ theo mã
from collections import defaultdict
groups = defaultdict(list)
for w in ["gin", "zen", "gig", "msg", "a", "et"]:
    groups[to_morse(w)].append(w)
for code, ws in groups.items():
    print(f"{code:<10} <- {ws}")

--...-.    <- ['gin', 'zen']
--...--.   <- ['gig', 'msg']
.-         <- ['a', 'et']


Biến thể 4 — **đếm số cặp từ cùng mã**: nếu một mã xuất hiện `c` lần thì có `c·(c-1)/2` cặp (chọn 2 trong `c`). Cộng qua mọi mã.

In [20]:
from collections import Counter
def count_same_code_pairs(words):
    cnt = Counter(to_morse(w) for w in words)
    return sum(c * (c - 1) // 2 for c in cnt.values())

print(count_same_code_pairs(["gin", "zen", "gig", "msg"]))      # 2 cặp: (gin,zen), (gig,msg)
print(count_same_code_pairs(["a", "et", "a"]))                   # 3 từ cùng mã -> 3 cặp

2
3


## 16. Bài tập tự luyện (TODO — đáp án ở cuối)

**Bài 1 (tính tay):** Viết mã Morse của `"sos"` và `"eee"`, `"ia"`. Chúng có trùng nhau không? Gán `answer_ex1 = (mã_sos, mã_eee, mã_ia, số_mã_khác_nhau_của_3_từ)`.

**Bài 2 (điền chỗ trống):** Hoàn thiện `to_morse_exercise(word)` dùng vòng `for` và `+=`.

**Bài 3 (viết lại):** `unique_morse_exercise(words)` dùng set comprehension.

**Bài 4 (biến thể):** `most_common_code(words)` trả về `(mã, số_lần)` xuất hiện nhiều nhất (hòa thì lấy mã gặp trước).

**Bài 5 (mini challenge):** `decode(morse_with_spaces)` giải mã chuỗi có **dấu cách** giữa các chữ, ví dụ `"... --- ..."` → `"sos"`.

In [21]:
answer_ex1 = None   # TODO

def to_morse_exercise(word: str) -> str:
    # TODO
    pass

def unique_morse_exercise(words: list[str]) -> int:
    # TODO
    pass

# assert to_morse_exercise("cab") == "-.-..--..."
# assert unique_morse_exercise(["gin", "zen", "gig", "msg"]) == 2

In [22]:
def most_common_code(words: list[str]) -> tuple[str, int]:
    # TODO: Counter
    pass

def decode(morse_with_spaces: str) -> str:
    # TODO: dict ngược mã -> chữ, split()
    pass

# assert most_common_code(["gin", "zen", "gig"]) == ("--...-.", 2)
# assert decode("... --- ...") == "sos"

### Gợi ý 3 mức

**Bài 1** — (1) Tra bảng: `s = ...`, `o = ---`, `e = .`, `i = ..`, `a = .-`. (2) `sos = ...---...`. (3) `eee` là ba mã `.` ghép lại; `ia` ghép `..` với `.-` — đếm kỹ số chấm/gạch rồi so ba chuỗi.

**Bài 2** — (1) Bắt đầu chuỗi rỗng. (2) Cộng mã từng chữ. (3) `code += MORSE[ord(c) - ord('a')]`.

**Bài 3** — (1) Set comprehension. (2) `{to_morse(w) for w in words}`. (3) `len(...)`.

**Bài 4** — (1) `Counter(to_morse(w) for w in words)`. (2) `most_common(1)` trả list một cặp. (3) `Counter` giữ thứ tự gặp khi hòa (Python ≥ 3.7).

**Bài 5** — (1) Đảo bảng: `{code: letter}`. (2) `morse.split()` tách theo dấu cách. (3) `"".join(rev[c] for c in ...)`.

---

## 17. Đáp án tham khảo

In [23]:
answer_ex1_solution = ("...---...", "...", "...-", 3)

def to_morse_exercise_solution(word):
    code = ""
    for c in word:
        code += MORSE[ord(c) - ord("a")]
    return code

def unique_morse_exercise_solution(words):
    return len({to_morse(w) for w in words})

def most_common_code_solution(words):
    return Counter(to_morse(w) for w in words).most_common(1)[0]

REV = {code: letter for letter, code in CODE.items()}     # dict comprehension: đảo key/value
def decode_solution(morse_with_spaces):
    return "".join(REV[c] for c in morse_with_spaces.split())

got = (to_morse("sos"), to_morse("eee"), to_morse("ia"), unique_morse(["sos", "eee", "ia"]))
print(got)
assert got == answer_ex1_solution
assert to_morse_exercise_solution("cab") == "-.-..--..."
assert unique_morse_exercise_solution(["gin", "zen", "gig", "msg"]) == 2
assert most_common_code_solution(["gin", "zen", "gig"]) == ("--...-.", 2)
assert decode_solution("... --- ...") == "sos"
assert decode_solution(" ".join(MORSE[ord(c) - 97] for c in "leetcode")) == "leetcode"
print("Đáp án PASS toàn bộ test!")

('...---...', '...', '...-', 3)
Đáp án PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist submit

```python
class Solution:
    def uniqueMorseRepresentations(self, words: list[str]) -> int:
        morse = [...]   # 26 mã cho sẵn trong đề
        return len({"".join(morse[ord(c) - ord("a")] for c in w) for w in words})
```

Mẫu chung: **biến đổi mỗi phần tử thành khóa → bỏ vào set → `len`**.

**Checklist trước khi submit:**
- [ ] Đếm mã khác nhau, **không** phải từ khác nhau?
- [ ] Nối mã **không** có dấu cách?
- [ ] Chỉ số `ord(c) - ord('a')` (không `+1`)?
- [ ] Dùng set (không list + `in`)?
- [ ] Đã thử `["gin","zen","gig","msg"] → 2`, `["a"] → 1`?